In [ ]:
# ============ CELL 1A: INSTALL PACKAGES ============

!pip uninstall -y transformers datasets bigframes cudf-polars-cu12 -q
!pip install transformers==4.44.0 datasets==2.21.0 accelerate==0.33.0 -q

In [ ]:
# ============ CELL 1B: IMPORT & CONFIGURATION ============
# Jalankan cell ini SETELAH restart kernel (jika diperlukan)

import torch
import pandas as pd
import numpy as np
import random
import os
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# KONFIGURASI - Sama dengan Skenario 5 untuk Fair Comparison
# ============================================================
EXPERIMENT_NAME = "RAW_DATA"  # Untuk penamaan file output

cfg = {
    "epochs": 8,
    "lr": 2e-5,
    "dropout": 0.2,
    "weight_decay": 0.02,
    "label_smooth": 0.1,
    "warmup": 0.1,
    "patience": 15,
}

# GPU Check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[INFO] Experiment: 5-Class {EXPERIMENT_NAME}")
print(f"[INFO] Config: {cfg['epochs']} epochs, LR {cfg['lr']}")
print(f"[INFO] Device: {device}")
if device == 'cuda':
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============ CELL 2: LOAD DATA RAW ============
# Upload file 'gojek_scraped_5class_all.csv' ke Kaggle Dataset terlebih dahulu

data_path = '/kaggle/input/gojek-raw-data/gojek_scraped_5class_all.csv'

print(f"[INFO] Loading RAW data (5-Class) from: {data_path}")
try:
    df = pd.read_csv(data_path)
except FileNotFoundError:
    print("[ERROR] File not found! Pastikan sudah upload dataset ke Kaggle.")
    raise

print(f"[INFO] Total data: {len(df):,}")
print(f"[INFO] Kolom: {list(df.columns)}")

# Label Encoding (5 Kelas)
label_map = {
    'very_negative': 0,
    'negative': 1,
    'neutral': 2,
    'positive': 3,
    'very_positive': 4
}
df['label'] = df['sentiment'].map(label_map)
df = df.dropna(subset=['text', 'label']).drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"\n[INFO] Distribusi SEBELUM split (Imbalanced):")
for label, count in df['sentiment'].value_counts().items():
    pct = count/len(df)*100
    print(f"  {label}: {count:,} ({pct:.1f}%)")

# Sample data untuk verifikasi slang masih ada
print(f"\n[INFO] Sample data (verifikasi slang masih ada):")
for i, row in df.head(3).iterrows():
    print(f"  [{row['sentiment']}] {row['text'][:80]}...")

In [ ]:
# ============ CELL 3: TRAIN/VAL/TEST SPLIT ============
# Stratified Split: 80% Train, 10% Val, 10% Test

# Step 1: Ambil 20% untuk Temp (Val + Test), sisa 80% untuk Train
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), 
    test_size=0.2, stratify=df['label'], random_state=42
)

# Step 2: Bagi Temp jadi 2 (50:50) -> 10% Val, 10% Test
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, 
    test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"[INFO] Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_texts):,}")
print(f"[INFO] Total: {len(train_texts) + len(val_texts) + len(test_texts):,}")

In [ ]:
# ============ CELL 4: TOKENIZATION ============
MODEL_NAME = 'indobenchmark/indobert-base-p1'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

# Create Datasets
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer)

print(f"[INFO] Tokenizer: {MODEL_NAME}")
print(f"[INFO] Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
# ============ CELL 5: MODEL & TRAINING ============
id2label = {0: 'Very Negative', 1: 'Negative', 2: 'Neutral', 3: 'Positive', 4: 'Very Positive'}
label2id = {v: k for k, v in id2label.items()}

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=cfg["dropout"],
    attention_probs_dropout_prob=cfg["dropout"]
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}

training_args = TrainingArguments(
    output_dir=f'./results_5class_{EXPERIMENT_NAME}',
    num_train_epochs=cfg['epochs'],
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=cfg['lr'],
    warmup_ratio=cfg["warmup"],
    lr_scheduler_type='cosine',
    weight_decay=cfg["weight_decay"],
    max_grad_norm=1.0,
    label_smoothing_factor=cfg["label_smooth"],
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg['patience'], early_stopping_threshold=0.001)]
)

print(f"[INFO] Starting Training - 5-Class {EXPERIMENT_NAME}")
print(f"[INFO] Epochs: {cfg['epochs']} | LR: {cfg['lr']} | Dropout: {cfg['dropout']}")
print(f"[INFO] Data: RAW (Slang belum dinormalisasi, Tidak Balance)")
trainer.train()
print("[INFO] Training Complete!")

In [ ]:
# ============ CELL 6: TRAINING CURVES ============
import matplotlib.pyplot as plt

# Extract logs
log_history = trainer.state.log_history

# Separate training loss and eval metrics
train_logs = [x for x in log_history if 'loss' in x and 'epoch' in x]
eval_logs = [x for x in log_history if 'eval_loss' in x and 'epoch' in x]

# Create DataFrames for easier plotting
df_train = pd.DataFrame(train_logs)
df_eval = pd.DataFrame(eval_logs)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot Loss
if not df_train.empty:
    axes[0].plot(df_train['epoch'], df_train['loss'], label='Training Loss', alpha=0.6)
if not df_eval.empty:
    axes[0].plot(df_eval['epoch'], df_eval['eval_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'Loss Curve - 5-Class {EXPERIMENT_NAME}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Accuracy/F1
if not df_eval.empty and 'eval_accuracy' in df_eval.columns:
    axes[1].plot(df_eval['epoch'], df_eval['eval_accuracy'], label='Val Accuracy', marker='o')
    if 'eval_f1' in df_eval.columns:
        axes[1].plot(df_eval['epoch'], df_eval['eval_f1'], label='Val F1 Score', marker='x')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Score')
    axes[1].set_title(f'Validation Metrics - 5-Class {EXPERIMENT_NAME}')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'training_history_5class_{EXPERIMENT_NAME}.png', dpi=150)
plt.show()

# Print Best Results
if not df_eval.empty:
    best_acc = df_eval['eval_accuracy'].max()
    best_f1 = df_eval['eval_f1'].max() if 'eval_f1' in df_eval.columns else 0
    min_loss = df_eval['eval_loss'].min()
    
    print(f"\n[INFO] Best Validation Accuracy: {best_acc:.4f}")
    print(f"[INFO] Best Validation F1: {best_f1:.4f}")
    print(f"[INFO] Lowest Validation Loss: {min_loss:.4f}")

In [ ]:
# ============ CELL 7: EVALUATION ============
import seaborn as sns

predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(-1)
y_true = predictions.label_ids

target_names = ['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive']

print("=" * 60)
print(f"TEST SET RESULTS - 5-Class {EXPERIMENT_NAME}")
print("=" * 60)
print(f"Accuracy:  {predictions.metrics['test_accuracy']:.4f} ({predictions.metrics['test_accuracy']*100:.2f}%)")
print(f"F1 Score:  {predictions.metrics['test_f1']:.4f}")
print(f"Precision: {predictions.metrics['test_precision']:.4f}")
print(f"Recall:    {predictions.metrics['test_recall']:.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=target_names))

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=target_names,
            yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - 5-Class {EXPERIMENT_NAME}')
plt.tight_layout()
plt.savefig(f'confusion_matrix_5class_{EXPERIMENT_NAME}.png', dpi=150)
plt.show()

In [ ]:
# ============ CELL 8: SAVE MODEL ============
import shutil

save_path = f'./saved_model_indobert_5class_{EXPERIMENT_NAME}'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

zip_name = f'indobert_5class_{EXPERIMENT_NAME}'
shutil.make_archive(zip_name, 'zip', save_path)
print(f"[INFO] Model saved to {save_path}")
print(f"[INFO] Zip file: {zip_name}.zip")